# LLC Quickstart in Google Colab

This notebook runs the same standard-library engineering core as the downloadable Quickstart. It is pinned to release v1.0.0; if the tagged repository or download is unavailable, upload the Quickstart ZIP when prompted.

In [ ]:
# Versioned publication inputs; the manual upload remains a fail-safe fallback.
REPOSITORY_URL = 'https://github.com/ai-native-power-electronics/llc-ai-workflow.git'
REPOSITORY_REF = 'v1.0.0'
QUICKSTART_URL = 'https://ainativepower.com/downloads/llc-ai-workflow-quickstart-v1.0.0.zip'

# Reader-editable engineering inputs
VIN_V = 400.0
VOUT_V = 36.0
POUT_W = 230.0
CANDIDATE_COUNT = 20

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import urllib.request
import zipfile

CONTENT = Path('/content')
SOURCE = CONTENT / 'llc_quickstart_source'
if SOURCE.exists():
    shutil.rmtree(SOURCE)

acquired = False
acquisition_errors = []
if REPOSITORY_URL:
    try:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF, REPOSITORY_URL, str(SOURCE)], check=True)
        acquired = True
    except subprocess.CalledProcessError as exc:
        acquisition_errors.append(f'tagged repository: {exc}')
if not acquired and QUICKSTART_URL:
    try:
        if SOURCE.exists():
            shutil.rmtree(SOURCE)
        archive = CONTENT / 'llc_quickstart.zip'
        urllib.request.urlretrieve(QUICKSTART_URL, archive)
        SOURCE.mkdir()
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(SOURCE)
        acquired = True
    except Exception as exc:
        acquisition_errors.append(f'versioned Quickstart: {exc}')
if not acquired:
    print('Automatic acquisition was unavailable:', *acquisition_errors, sep='\n- ')
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one LLC Quickstart ZIP.')
    archive = CONTENT / next(iter(uploaded))
    archive.write_bytes(next(iter(uploaded.values())))
    if SOURCE.exists():
        shutil.rmtree(SOURCE)
    SOURCE.mkdir()
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(SOURCE)

def resolve_package_root(source):
    package_candidates = [
        path.parents[1]
        for path in source.rglob('llc_tool/__init__.py')
    ]
    if len(package_candidates) != 1:
        raise RuntimeError(f'Expected one package root, found {package_candidates!r}')
    package_root = package_candidates[0]
    if not (package_root / 'configs' / 'quick_demo.json').is_file():
        raise RuntimeError(f'Package root was detected incorrectly: {package_root}')
    return package_root

PACKAGE_ROOT = resolve_package_root(SOURCE)
sys.path.insert(0, str(PACKAGE_ROOT))
print(f'Package root: {PACKAGE_ROOT}')

In [ ]:
import copy
import json
from datetime import datetime, timezone
from llc_tool.config import load_and_validate_config, validate_config
from llc_tool.workflow import run_workflow

config = load_and_validate_config(PACKAGE_ROOT / 'configs' / 'quick_demo.json')
config = copy.deepcopy(config)
config['input_spec']['vin_v'] = VIN_V
config['input_spec']['vout_v'] = VOUT_V
config['input_spec']['pout_w'] = POUT_W
config['candidate_count'] = CANDIDATE_COUNT
config = validate_config(config)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT = CONTENT / 'LLC_Colab_Runs' / f'demo-{stamp}'
result = run_workflow(config, OUTPUT)
print(json.dumps(result, indent=2, sort_keys=True))

In [ ]:
from IPython.display import Markdown, SVG, display
from google.colab import files

summary = json.loads((OUTPUT / 'summary.json').read_text(encoding='utf-8'))
verification = json.loads((OUTPUT / 'verification.json').read_text(encoding='utf-8'))
cases = json.loads((OUTPUT / 'pedagogical_cases.json').read_text(encoding='utf-8'))
display(Markdown(f"**Output integrity:** {verification['status']}  \
**Gate counts:** {summary['gate_counts']}"))
display(SVG(filename=str(OUTPUT / 'plots' / 'gate_counts.svg')))
table = ['| Candidate | Case | FHA gate | Time-domain | Final disposition |', '|---|---|---|---|---|']
for case in cases:
    record = case['record']
    table.append('| {candidate} | {case_type} | {fha} | {time_domain} | {final} |'.format(
        candidate=record['candidate_id'],
        case_type=case['case_type'],
        fha=record.get('fha_screen', {}).get('gate_result', 'not_applicable'),
        time_domain=record.get('time_domain_evaluation', {}).get('execution_status', 'not_applicable'),
        final=record['decision_contract']['gate_result'],
    ))
display(Markdown('## Pass, reject and failed cases\n\n' + '\n'.join(table)))
for filename, title in (('fha_pass_case.svg', 'FHA pass case'), ('fha_reject_case.svg', 'FHA reject case')):
    plot = OUTPUT / 'plots' / filename
    if plot.is_file():
        display(Markdown(f'### {title}'))
        display(SVG(filename=str(plot)))
display(Markdown('## Continue the evidence-led sequence\n\n**AI-Native Power Electronics — Rafael Collado**  \n[Get the next Field Note](https://ainativepower.com/field-notes)'))
download_base = CONTENT / f'llc-demo-{stamp}'
download_zip = Path(shutil.make_archive(str(download_base), 'zip', OUTPUT))
print('Downloading the complete report bundle...')
files.download(str(download_zip))